# Entregable 2: recuperación manual de una campaña CUDA

Este notebook recupera la campaña `e2-cuda-selection-v1` después de perder la caché de Colab. Conserva el motor científico A3 y su registro originales. Las tres ayudas de recuperación se descargan en otra carpeta y no se copian dentro del motor.

El caso autorizado conserva **494 resultados confirmados y un único trabajo pendiente**. El resultado, el tiempo y los pasos del intento físico perdido permanecen **desconocidos**. Una nueva ejecución observada, con el mismo trabajo y semilla, puede reconciliar ese trabajo lógico; no revela qué ocurrió en el intento perdido.

1. Con `RUN_SEARCH=False` puedes revisar el plan en CPU: no se usan secretos, no se instala CUDA, no se descargan precios ni se reclama un recibo de ejecución.
2. Para ejecutar, necesitas una **Tesla T4** disponible en Colab. Su cuota pertenece al proveedor y es independiente de los límites operativos de 8 horas por sesión y 24 horas globales. Este notebook no reserva una GPU, no contrata recursos y no estima cuándo se renovará la cuota.
3. Habilita el acceso al secreto `GITHUB_TOKEN` para **este nuevo notebook**; puedes conservar el mismo token válido del repositorio.
4. Sólo cuando vayas a ejecutar la recuperación autorizada cambia **`RUN_SEARCH=True` y `RETRY_LOST_ONCE=True`** y ejecuta las celdas en orden. El permiso corresponde a un único reintento físico de ese trabajo, no a una política de reintentos automáticos.

Los 494 trabajos confirmados se reutilizan. Si el nuevo resultado quedó confirmado externamente pero se perdió la confirmación del registro principal, se recupera ese recibo sin otro ajuste GPU. Si vuelve a perderse un intento sin recibo observado, la campaña se bloquea para revisar el caso. La asignación conservadora de la sesión perdida es de 8 horas; el límite global permanece en 24 horas, sin ampliación. No se leen valores finales de 2025–2026 ni se fija R.


In [ ]:
from pathlib import Path
import os, re, sys, json, hashlib, subprocess, platform, shutil, math

SOURCE_COMMIT = "564ba4762f84a800b02fa4c734f5752a169f5223"
EXPECTED_SEARCH_REGISTRATION_SHA256 = "78c3c5b8ca042a347aaa80d642f3affd8b5911e8697d8747a2de915d1c3194a9"
RECOVERY_COMMIT = "b2d8cc9ca196f08670c1121efb89549c1838e38d"
EXPECTED_RECOVERY_AUTHORIZATION_SHA256 = "7545da0cd424ebe84b3f5adb415849d1521711e47632084ca678ef09ed9b7fea"
DATA_COMMIT = "0b079b50625a936456a70695532853fc13298f31"
REPOSITORY_URL = "https://github.com/jassan-bit/DeepLearning_202630.git"

RUN_SEARCH = False
RETRY_LOST_ONCE = False
SHOW_PLAN = True
MODELS = None  # La recuperación conserva los modelos del plan sellado.
ORIGINS = None  # La recuperación conserva los orígenes del plan sellado.
DEVICE = "cuda:0"
SESSION_HOURS = 8
CAMPAIGN_HOURS = 24
STORAGE_GIB = 10
CAMPAIGN_NAME = "e2-cuda-selection-v1"
GITHUB_REPOSITORY = "jassan-bit/DeepLearning_202630"
GITHUB_BRANCH_PREFIX = "campaigns"
GITHUB_PATH_PREFIX = "campaigns"

# Reutiliza el entorno y la caché del piloto; mantiene separado el código nuevo.
WORKSPACE = Path.cwd()/"entregable2_gpu_workspace"
REPO = WORKSPACE/"search-source"
RECOVERY_REPO = WORKSPACE/"recovery-source"
VENV = WORKSPACE/"venv-py312-cu126"
DATA_CACHE = WORKSPACE/"development_cache"
LOCAL_CAMPAIGN_CACHE = WORKSPACE/"search-campaigns"
UV_VERSION = "0.9.5"
MANAGED_PYTHON_VERSION = "3.12.12"
PIP_VERSION = "25.3"
UV_WHEEL_SHA256 = "6507bbbcd788553ec4ad5a96fa19364dc0f58b023e31d79868773559a83ec181"
PIP_WHEEL_SHA256 = "9655943313a94722b7774661c21049070f6bbb0a1516bf02f7c8d5d9201514cd"

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
CUDA_ENV = dict(os.environ)
CUDA_ENV.update(CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONNOUSERSITE="1",
                OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
CUDA_ENV.pop("PYTHONPATH", None)
CUDA_ENV.pop("GITHUB_TOKEN", None)
CUDA_ENV.pop("GH_TOKEN", None)
CUDA_ENV.update(UV_CACHE_DIR=str(WORKSPACE/"uv-cache"),
                UV_PYTHON_INSTALL_DIR=str(WORKSPACE/"managed-python"),
                UV_PYTHON_BIN_DIR=str(WORKSPACE/"managed-python-bin"), UV_NATIVE_TLS="true")
SOURCE_READY = False
RECOVERY_SOURCE_READY = False
PLAN_READY = False
REMOTE_READY = False
HARDWARE_READY = False
GPU_READY = False
DATA_READY = False
SYNTHETIC_READY = False
SEARCH_PLAN = None
EXECUTION_ATTEMPTED = False
LAST_SEARCH_RETURN_CODE = None
CAMPAIGN_OUTPUT = None
GITHUB_ENV = None

def run(args, *, cwd=None, capture=False, input_text=None):
    return subprocess.run([str(x) for x in args], cwd=cwd, env=CUDA_ENV, check=True,
                          text=True, input=input_text, capture_output=capture)

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()


## 1. Descargar dos fuentes separadas y verificadas

El motor científico conserva exactamente el commit A3 `564ba4762f84a800b02fa4c734f5752a169f5223`, sus 61 hashes y el registro original. El segundo checkout contiene sólo las tres ayudas de recuperación, sus pruebas y la autorización fijada por SHA256. Los CSV no forman parte de ninguno de los checkouts. Las ayudas permanecen fuera de `search-source/scripts`, para conservar intacta la verificación del motor.


In [ ]:
RECOVERY_SOURCE_READY = False
if not re.fullmatch(r"[0-9a-f]{40}", RECOVERY_COMMIT):
    raise RuntimeError("Falta el commit publicado de recuperación; no usar main ni un marcador.")
if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_RECOVERY_AUTHORIZATION_SHA256):
    raise RuntimeError("Falta el SHA256 publicado de la autorización de recuperación.")
SPARSE_PATHS = [
    "/scripts/", "/tests/", "/requirements-e2-training-gpu.lock.txt",
    "/book_e2/_static/search_protocol_20261109/protocol.json",
    "/experiments/dl_r2_cv_20261104/protocol.json",
    "/experiments/e2_development_pilot_20261109/source_protocol.json",
    "/experiments/e2_gpu_pilot_20261109/", "/experiments/e2_cuda_search_20261109/",
]
SOURCE_READY = False
if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
    raise RuntimeError("Falta el commit publicado de 40 caracteres; no usar main ni un marcador.")
if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("El entorno fijado corresponde a Linux x86_64.")
if not REPO.exists():
    REPO.mkdir(parents=True)
    run(["git", "init", REPO])
    run(["git", "-C", REPO, "remote", "add", "origin", REPOSITORY_URL])
    run(["git", "-C", REPO, "config", "core.sparseCheckout", "true"])
    run(["git", "-C", REPO, "config", "remote.origin.promisor", "true"])
    run(["git", "-C", REPO, "config", "remote.origin.partialclonefilter", "blob:none"])
    run(["git", "-C", REPO, "sparse-checkout", "set", "--no-cone", "--stdin"],
        input_text="\n".join(SPARSE_PATHS)+"\n")
    run(["git", "-C", REPO, "fetch", "--filter=blob:none", "--depth=1", "origin", SOURCE_COMMIT])
    run(["git", "-C", REPO, "checkout", "--detach", "FETCH_HEAD"])
if run(["git", "-C", REPO, "rev-parse", "HEAD"], capture=True).stdout.strip() != SOURCE_COMMIT:
    raise RuntimeError("El checkout existente pertenece a otro commit; usa un WORKSPACE nuevo.")
if (REPO/"results").exists():
    raise RuntimeError("Este checkout debe contener sólo código; usa una carpeta nueva sin results.")
SOURCE_READY = True
print("Fuente verificada:", SOURCE_COMMIT)

RECOVERY_SPARSE_PATHS = [
    "/scripts/e2_manual_reconciliation.py", "/scripts/e2_recovery_receipts.py",
    "/scripts/run_e2_cuda_recovery.py", "/tests/test_e2_manual_reconciliation.py",
    "/tests/test_e2_recovery_receipts.py", "/tests/test_e2_cuda_recovery.py",
    "/experiments/e2_recovery_20261110/authorization.json",
]
if not RECOVERY_REPO.exists():
    RECOVERY_REPO.mkdir(parents=True)
    run(["git", "init", RECOVERY_REPO])
    run(["git", "-C", RECOVERY_REPO, "remote", "add", "origin", REPOSITORY_URL])
    run(["git", "-C", RECOVERY_REPO, "config", "core.sparseCheckout", "true"])
    run(["git", "-C", RECOVERY_REPO, "config", "remote.origin.promisor", "true"])
    run(["git", "-C", RECOVERY_REPO, "config", "remote.origin.partialclonefilter", "blob:none"])
    run(["git", "-C", RECOVERY_REPO, "sparse-checkout", "set", "--no-cone", "--stdin"],
        input_text="\n".join(RECOVERY_SPARSE_PATHS)+"\n")
    run(["git", "-C", RECOVERY_REPO, "fetch", "--filter=blob:none", "--depth=1", "origin", RECOVERY_COMMIT])
    run(["git", "-C", RECOVERY_REPO, "checkout", "--detach", "FETCH_HEAD"])
if run(["git", "-C", RECOVERY_REPO, "rev-parse", "HEAD"], capture=True).stdout.strip() != RECOVERY_COMMIT:
    raise RuntimeError("La carpeta de recuperación pertenece a otra versión; conserva la campaña y usa otro checkout.")
if (RECOVERY_REPO/"results").exists():
    raise RuntimeError("La carpeta de recuperación no debe incluir resultados ni datos de mercado.")
RECOVERY_SOURCE_READY = True
print("Ayudas de recuperación verificadas:", RECOVERY_COMMIT)


## 2. Inspeccionar la recuperación en CPU

Esta inspección no reclama recibos, no restaura resultados remotos ni inicia un ajuste. Valida la autorización y muestra el caso que resolverá el controlador. El motor, los candidatos, los orígenes, las semillas y la receta permanecen fijados por el plan original.

Los límites siguen siendo 8 horas por sesión y 24 horas globales. La sesión perdida conserva una asignación conservadora de 8 horas, con unas 16 horas globales antes de contabilizar el nuevo consumo. Esa asignación no transforma en conocidos el tiempo ni los pasos del intento perdido. Esta autorización no amplía el presupuesto.


In [ ]:
PLAN_READY = False
SEARCH_PLAN = None
if not SOURCE_READY or not RECOVERY_SOURCE_READY:
    raise RuntimeError("Verifica primero la fuente congelada.")
if not re.fullmatch(r"[A-Za-z0-9_.-]{1,80}", CAMPAIGN_NAME) or CAMPAIGN_NAME in {".", ".."}:
    raise ValueError("Elige un nombre simple de campaña, sin rutas ni espacios.")
for value, label in [(SESSION_HOURS, "SESSION_HOURS"), (CAMPAIGN_HOURS, "CAMPAIGN_HOURS"), (STORAGE_GIB, "STORAGE_GIB")]:
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:
        raise ValueError(label+" debe ser un número finito positivo.")
if SESSION_HOURS > CAMPAIGN_HOURS:
    raise ValueError("El límite de sesión no puede superar el límite global.")
if SESSION_HOURS != 8 or CAMPAIGN_HOURS != 24:
    raise ValueError("Esta recuperación conserva los límites originales 8/24; no autoriza ampliar el presupuesto.")
if MODELS is not None or ORIGINS is not None:
    raise ValueError("La recuperación utiliza los modelos y orígenes del plan sellado, sin seleccionar otro subconjunto.")
CAMPAIGN_OUTPUT = LOCAL_CAMPAIGN_CACHE/CAMPAIGN_NAME
GITHUB_BRANCH = GITHUB_BRANCH_PREFIX+"/"+CAMPAIGN_NAME
GITHUB_PREFIX = GITHUB_PATH_PREFIX+"/"+CAMPAIGN_NAME
PROTOCOL_RELATIVE = "book_e2/_static/search_protocol_20261109/protocol.json"
NORMALIZED_PROTOCOL_SHA256 = "e0030c990a4afa15665aec254211b0d9e680e9ac5d92dfb8094e43e965152abe"
AUTHORIZATION = RECOVERY_REPO/"experiments/e2_recovery_20261110/authorization.json"

def verify_source_snapshot(*, check_protocol=True):
    changed = run(["git", "-C", REPO, "diff", "--name-only", "HEAD"], capture=True).stdout.splitlines()
    if any(name != PROTOCOL_RELATIVE for name in changed):
        raise RuntimeError("Hay cambios en la fuente congelada; conserva la campaña y revisa el checkout.")
    untracked = run(["git", "-C", REPO, "ls-files", "--others", "--exclude-standard", "scripts"], capture=True).stdout.splitlines()
    if any(name.endswith(".py") for name in untracked):
        raise RuntimeError("Hay módulos Python no registrados en scripts.")
    if check_protocol and sha256_file(REPO/PROTOCOL_RELATIVE) != NORMALIZED_PROTOCOL_SHA256:
        raise RuntimeError("El protocolo cambió respecto de los bytes registrados.")

def verify_controller_snapshot():
    head = run(["git", "-C", RECOVERY_REPO, "rev-parse", "HEAD"], capture=True).stdout.strip()
    if head != RECOVERY_COMMIT:
        raise RuntimeError("Cambió el commit del controlador de recuperación.")
    if run(["git", "-C", RECOVERY_REPO, "diff", "--name-only", "HEAD"], capture=True).stdout.strip():
        raise RuntimeError("Hay cambios en el controlador congelado de recuperación.")
    untracked = run(["git", "-C", RECOVERY_REPO, "ls-files", "--others", "--exclude-standard", "scripts"], capture=True).stdout.splitlines()
    if any(name.endswith(".py") for name in untracked):
        raise RuntimeError("Hay módulos no registrados en la carpeta de recuperación.")
    if sha256_file(AUTHORIZATION) != EXPECTED_RECOVERY_AUTHORIZATION_SHA256:
        raise RuntimeError("La autorización no coincide con el SHA256 publicado.")

verify_controller_snapshot()
verify_source_snapshot(check_protocol=False)
run([sys.executable, REPO/"scripts/prepare_e2_pilot_snapshot.py", "--root", REPO], cwd=REPO)
if sha256_file(REPO/PROTOCOL_RELATIVE) != NORMALIZED_PROTOCOL_SHA256:
    raise RuntimeError("El protocolo no coincide con los bytes registrados.")

def search_command(python, *, execute):
    command = [python, RECOVERY_REPO/"scripts/run_e2_cuda_recovery.py", "--execute" if execute else "--plan-only",
               "--engine-root", REPO, "--controller-source-commit", RECOVERY_COMMIT,
               "--authorization", AUTHORIZATION, "--expected-authorization-sha256", EXPECTED_RECOVERY_AUTHORIZATION_SHA256,
               "--source-commit", SOURCE_COMMIT, "--expected-registration-sha256", EXPECTED_SEARCH_REGISTRATION_SHA256,
               "--output", CAMPAIGN_OUTPUT, "--session-hours", SESSION_HOURS,
               "--campaign-hours", CAMPAIGN_HOURS, "--storage-gib", STORAGE_GIB, "--device", DEVICE,
               "--github-repository", GITHUB_REPOSITORY, "--github-branch", GITHUB_BRANCH,
               "--github-prefix", GITHUB_PREFIX]
    if MODELS is not None:
        command += ["--models", *MODELS]
    if ORIGINS is not None:
        command += ["--origins", *ORIGINS]
    if execute:
        if RETRY_LOST_ONCE:
            command.append("--retry-lost-once")
        command += ["--data-root", DATA_CACHE,
                    "--source-root-itransformer", REPO/"work/author_sources/iTransformer"]
    return command

if SHOW_PLAN or RUN_SEARCH:
    planned = run(search_command(sys.executable, execute=False), cwd=RECOVERY_REPO, capture=True)
    SEARCH_PLAN = json.loads(planned.stdout)
    print(json.dumps(SEARCH_PLAN, ensure_ascii=False, indent=2, allow_nan=False))
    PLAN_READY = True
else:
    print("Activa SHOW_PLAN para revisar el dominio antes del lanzamiento.")


## 3. Habilitar GitHub para este nuevo notebook

La campaña se guarda en una rama dedicada de **`jassan-bit/DeepLearning_202630`**. El disco de Colab es sólo una caché recuperable. El modo de planificación no usa credenciales ni escribe en el repositorio.

Para ejecutar desde Colab, crea un [token personal de GitHub con alcance fino](https://github.com/settings/personal-access-tokens/new) con acceso al repositorio elegido y permiso **Contents: read and write**. En el panel **Secretos** de Colab, añade `GITHUB_TOKEN` y habilita su acceso para este notebook. Si el secreto ya existe, puedes reutilizar el mismo token: habilita su acceso para este nuevo notebook. También puede recibirse como variable de entorno en un servidor propio. No pegues el token en una celda, un archivo de configuración o un mensaje.

La lectura del código público no necesita ese token. Sólo la sincronización de la campaña lo utiliza. Se verifica el acceso al repositorio; el ejecutor exige además confirmar en GitHub el estado de inicio antes de entrenar y el resultado antes de avanzar al siguiente trabajo. Un error de sincronización pausa la campaña. Los JSON autorizados no contienen credenciales, rutas internas ni pesos de modelos.

El recibo físico de esta intervención se conserva aparte, en la rama `recoveries/e2-cuda-selection-v1/20261110`, con JSON de creación exclusiva. El registro principal conserva su rama original y los 494 resultados confirmados.


In [ ]:
REMOTE_READY = False
GITHUB_ENV = None

def github_execution_environment():
    import urllib.request, urllib.error
    if GITHUB_REPOSITORY != "jassan-bit/DeepLearning_202630":
        raise RuntimeError("Este notebook fija el repositorio autorizado para la campaña.")
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN").strip()
        except Exception:
            raise RuntimeError("Añade GITHUB_TOKEN en Secretos de Colab y habilita su acceso al notebook.") from None
    if not token or any(character.isspace() for character in token):
        raise RuntimeError("El secreto GITHUB_TOKEN está vacío o tiene un formato inválido.")
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, request, response, code, message, headers, new_url):
            return None
    request = urllib.request.Request("https://api.github.com/repos/"+GITHUB_REPOSITORY,
        headers={"Authorization":"Bearer "+token, "Accept":"application/vnd.github+json",
                 "X-GitHub-Api-Version":"2022-11-28"}, method="GET")
    try:
        with urllib.request.build_opener(NoRedirect()).open(request, timeout=30) as response:
            raw = response.read(65537)
            if len(raw) > 65536:
                raise ValueError("Unexpected response size")
            payload = json.loads(raw)
    except urllib.error.HTTPError as error:
        raise RuntimeError("GitHub rechazó la comprobación de acceso (HTTP "+str(error.code)+"). Revisa el secreto y su repositorio autorizado.") from None
    except Exception:
        raise RuntimeError("No se pudo comprobar el acceso a GitHub; no se habilitó el entrenamiento.") from None
    if not isinstance(payload, dict) or payload.get("full_name", "").lower() != GITHUB_REPOSITORY.lower():
        raise RuntimeError("GitHub no confirmó el repositorio esperado.")
    environment = dict(CUDA_ENV)
    environment["GITHUB_TOKEN"] = token
    return environment

if RUN_SEARCH:
    if not RETRY_LOST_ONCE:
        raise RuntimeError("La recuperación requiere activar explícitamente RETRY_LOST_ONCE para el único trabajo autorizado.")
    if not PLAN_READY:
        raise RuntimeError("Revisa primero el plan sin entrenamiento.")
    GITHUB_ENV = github_execution_environment()
    REMOTE_READY = True
    print("Acceso al repositorio confirmado. El ejecutor comprobará el guardado remoto antes de cada ajuste.")
else:
    print("Planificación: no se leyeron secretos ni se escribió en GitHub.")


## 4. Comprobar una Tesla T4 antes de reclamar la recuperación

Si Colab muestra un límite de uso de GPU, conserva `RUN_SEARCH=False` y revisa el plan en CPU. La disponibilidad y la renovación de cuota dependen de Colab; los límites 8/24 del ejecutor no reservan acceso a una tarjeta ni indican cuándo volverá a estar disponible. No se activa una GPU de pago ni otra clase de tarjeta automáticamente.

La ejecución exige una Tesla T4 visible y una operación CUDA válida con el entorno fijado. Estas comprobaciones ocurren antes de que el controlador pueda reclamar el único nuevo intento físico.


In [ ]:
HARDWARE_READY = False
GPU_READY = False
DATA_READY = False
SYNTHETIC_READY = False
if RUN_SEARCH:
    if not REMOTE_READY or not PLAN_READY:
        raise RuntimeError("Verifica el plan y el guardado en GitHub antes de instalar CUDA.")
    if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
        raise RuntimeError("SOURCE_COMMIT debe contener el SHA GPU publicado de 40 caracteres; no usar main.")
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("El lock CUDA corresponde a Linux x86_64.")
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("No hay GPU NVIDIA visible. Activa GPU en el proveedor antes de continuar.")
    probe = run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture=True)
    if not probe.stdout.strip():
        raise RuntimeError("nvidia-smi no informó ninguna GPU.")
    names = [line.split(",", 1)[0].strip() for line in probe.stdout.splitlines() if line.strip()]
    if not names or names[0] != "Tesla T4":
        raise RuntimeError("La campaña sellada requiere una Tesla T4. Revisa la disponibilidad del proveedor antes de continuar.")
    print(probe.stdout.strip())
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    HARDWARE_READY = True
    print("Python del kernel:", ".".join(map(str, sys.version_info[:3])))
else:
    print("Planificación: no se consultó ni inicializó una GPU.")


## 5. Reutilizar el Python aislado y comprobar CUDA

Se reutiliza el entorno del piloto si sus versiones coinciden. El bootstrap fija uv 0.9.5, CPython 3.12.12 y pip 25.3, comprueba los wheels por SHA256 e instala el lock PyTorch 2.8.0 + CUDA 12.6. No modifica paquetes del kernel anfitrión. La instalación y las pruebas previas no forman parte del cronómetro del ejecutor; tampoco definen el tiempo cobrado o la cuota de GPU del proveedor.


In [ ]:
GPU_READY = False
DATA_READY = False
SYNTHETIC_READY = False
if RUN_SEARCH:
    if not HARDWARE_READY:
        raise RuntimeError("Comprueba primero Linux y la GPU NVIDIA en la sección 1.")
    
    def bootstrap_python312():
        bootstrap = WORKSPACE/"bootstrap"
        bootstrap.mkdir(parents=True, exist_ok=True)
        uv_home = bootstrap/("uv-"+UV_VERSION)
        uv_binary = uv_home/"bin/uv"
        uv_requirements = bootstrap/"uv.requirements.txt"
        uv_requirements.write_text("uv=="+UV_VERSION+" --hash=sha256:"+UV_WHEEL_SHA256+"\n", encoding="utf-8")
        if not uv_binary.is_file():
            if uv_home.exists():
                raise RuntimeError("El bootstrap uv está incompleto; elige un WORKSPACE nuevo.")
            run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-cache-dir",
                 "--no-deps", "--only-binary=:all:", "--require-hashes", "--index-url", "https://pypi.org/simple",
                 "--target", uv_home, "-r", uv_requirements])
        uv_version = run([uv_binary, "--version"], capture=True).stdout.strip().split()
        if uv_version[:2] != ["uv", UV_VERSION]:
            raise RuntimeError("La versión local de uv no coincide con la fijada.")
        managed_directory = WORKSPACE/"managed-python"
        run([uv_binary, "python", "install", MANAGED_PYTHON_VERSION,
             "--install-dir", managed_directory, "--no-bin", "--no-config"])
        managed_python = Path(run([uv_binary, "python", "find", MANAGED_PYTHON_VERSION,
                                  "--managed-python", "--no-python-downloads", "--no-project", "--no-config"],
                                 capture=True).stdout.strip())
        if managed_directory.resolve() not in managed_python.resolve().parents:
            raise RuntimeError("uv seleccionó un Python fuera del directorio gestionado.")
        if not (VENV/"bin/python").is_file():
            run([uv_binary, "venv", "--python", managed_python, "--no-python-downloads", "--no-project",
                 "--no-config", VENV])
        python = VENV/"bin/python"
        info = json.loads(run([python, "-c", "import sys,json; print(json.dumps({'version':list(sys.version_info[:3])}))"],
                              capture=True).stdout)
        if info["version"] != [3, 12, 12]:
            raise RuntimeError("El venv no usa CPython 3.12.12; elige una carpeta nueva.")
        pip_requirements = bootstrap/"pip.requirements.txt"
        pip_requirements.write_text("pip=="+PIP_VERSION+" --hash=sha256:"+PIP_WHEEL_SHA256+"\n", encoding="utf-8")
        run([uv_binary, "pip", "install", "--python", python, "--no-deps", "--only-binary=:all:",
             "--require-hashes", "--index-url", "https://pypi.org/simple", "--no-config", "-r", pip_requirements])
        run([python, "-c", "import pip; assert pip.__version__=='25.3'"])
        print("Python del kernel:", ".".join(map(str, sys.version_info[:3])))
        print("Python aislado del buscador:", ".".join(map(str, info["version"])), "con pip", PIP_VERSION)
        return python
    
    GPU_PYTHON = bootstrap_python312()
    run([GPU_PYTHON, "-m", "pip", "install", "-r", REPO/"requirements-e2-training-gpu.lock.txt"])
    run([GPU_PYTHON, "-m", "pip", "check"])
    run([GPU_PYTHON, "-c", "import sys,torch; assert sys.version_info[:3]==(3,12,12); assert torch.__version__=='2.8.0+cu126'; assert torch.version.cuda=='12.6'; print('Python/Torch/CUDA: versiones verificadas')"])
    # Sólo la prueba CUDA real habilita las siguientes celdas; no se abren datos aquí.
    run([GPU_PYTHON, REPO/"scripts/run_e2_gpu_pilot.py", "--check-environment", "--device", DEVICE], cwd=REPO)
    GPU_READY = True
else:
    print("Planificación: no se instalaron paquetes ni se ejecutaron kernels CUDA.")


## 6. Fuentes y pruebas artificiales antes de los CSV

Se verifican las ocho fuentes oficiales de iTransformer. Una caché válida puede reutilizarse; los archivos o recibos diferentes se conservan y bloquean el lanzamiento. Las pruebas artificiales de CUDA y del adaptador se ejecutan antes de descargar observaciones. Sus ajustes sintéticos no son resultados HPO/NAS sobre mercado.


In [ ]:
SYNTHETIC_READY = False
DATA_READY = False
def ensure_itransformer_sources():
    import ast
    source = REPO/"scripts/fetch_e2_itransformer_sources.py"
    output = REPO/"work/author_sources/iTransformer"
    def reject_symlinks(path):
        if any(part.is_symlink() for part in (path.absolute(), *path.absolute().parents)):
            raise RuntimeError("El árbol iTransformer no admite enlaces simbólicos.")
    reject_symlinks(source)
    reject_symlinks(output)
    frozen = run(["git", "-C", REPO, "show", SOURCE_COMMIT+":scripts/fetch_e2_itransformer_sources.py"], capture=True).stdout
    if source.read_bytes() != frozen.encode():
        raise RuntimeError("El descargador iTransformer difiere de la fuente congelada.")
    constants = {node.targets[0].id: ast.literal_eval(node.value)
                 for node in ast.parse(frozen).body if isinstance(node, ast.Assign)
                 and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name)
                 and node.targets[0].id == "MANIFEST"}
    manifest, maximum = constants["MANIFEST"], 1024*1024
    if not output.exists():
        run([GPU_PYTHON, source, "--output", "work/author_sources/iTransformer"], cwd=REPO)
    if not output.is_dir():
        raise RuntimeError("El destino iTransformer no es un directorio válido.")
    for path in output.rglob("*"):
        reject_symlinks(path)
        relative = path.relative_to(output).as_posix()
        if path.is_file() and path.suffix in {".py", ".pyc", ".pyo", ".pth", ".so"} and relative not in manifest["files"]:
            if "__pycache__" not in path.relative_to(output).parts or path.suffix != ".pyc":
                raise RuntimeError("Hay código no registrado en el árbol iTransformer.")
    for relative, expected in manifest["files"].items():
        path = output/relative
        reject_symlinks(path)
        if not path.is_file() or path.stat().st_size > maximum or sha256_file(path) != expected:
            raise RuntimeError("Fuente iTransformer ausente o con hash distinto: "+relative)
    receipt = output/"source_receipt.json"
    reject_symlinks(receipt)
    expected_receipt = {"status":"complete", "repository":manifest["repository"], "commit":manifest["commit"],
                        "verified_files":manifest["files"], "weights_downloaded":False, "market_data_read":False}
    if not receipt.is_file() or receipt.stat().st_size > maximum or json.loads(receipt.read_text()) != expected_receipt:
        raise RuntimeError("El recibo iTransformer no coincide con los ocho archivos registrados.")
    print("iTransformer: ocho fuentes verificadas; caché reutilizable.")

if RUN_SEARCH:
    if not GPU_READY or not REMOTE_READY:
        raise RuntimeError("Comprueba CUDA y el acceso remoto antes de los tests.")
    ensure_itransformer_sources()
    smoke_env = dict(CUDA_ENV)
    smoke_env["E2_ITRANSFORMER_SOURCE_ROOT"] = str(REPO/"work/author_sources/iTransformer")
    for pattern in ["test_e2_cuda_models.py", "test_e2_cuda_training.py", "test_e2_cuda_bindings.py", "test_e2_search_session.py"]:
        subprocess.run([str(GPU_PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"],
                       cwd=REPO, env=smoke_env, check=True)
    recovery_test_env = dict(smoke_env)
    recovery_test_env["PYTHONPATH"] = os.pathsep.join([str(RECOVERY_REPO/"scripts"), str(REPO/"scripts")])
    recovery_test_env["E2_ENGINE_ROOT"] = str(REPO)
    for pattern in ["test_e2_manual_reconciliation.py", "test_e2_recovery_receipts.py", "test_e2_cuda_recovery.py"]:
        subprocess.run([str(GPU_PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"],
                       cwd=RECOVERY_REPO, env=recovery_test_env, check=True)
    SYNTHETIC_READY = True
    print("Comprobaciones artificiales completadas antes del acceso al desarrollo.")
else:
    print("Planificación: no se descargaron fuentes de autor ni se ajustaron modelos sintéticos.")


## 7. Obtener únicamente el desarrollo registrado

Con la ejecución desactivada no se descargan CSV. Al activarla se admiten sólo los derivados de 2023–2024 y su protocolo fijados por hash. El helper puede reutilizar la caché existente y rechaza bytes distintos. La recuperación conserva los once orígenes y los modelos declarados en el plan sellado. El origen 2025-01-01 es un marcador de calendario que utiliza sólo desarrollo disponible hasta 2024; sus valores finales no se abren.


In [ ]:
DATA_READY = False
DATA_FILES = {
    "results/eda_development/daily_market_development.csv": "bf6bb5160af924aef6a15c3684aeae932df3017e8639888bb7e91ce787900171",
    "results/eda_development/daily_volatility_development.csv": "20b96374a659eb87dadfa68aa3d6b6b5fb6bea5b10eebe6f2c186ad69861844a",
}
if RUN_SEARCH:
    if not GPU_READY or not PLAN_READY or not REMOTE_READY or not SYNTHETIC_READY:
        raise RuntimeError("Verifica primero el plan, CUDA, las pruebas artificiales y el guardado en GitHub.")
    run([GPU_PYTHON, REPO/"scripts/fetch_e2_development_sources.py", "--output", DATA_CACHE], cwd=REPO)
    for relative, checksum in DATA_FILES.items():
        if sha256_file(DATA_CACHE/relative) != checksum:
            raise RuntimeError("El derivado no coincide con el SHA256 registrado.")
    DATA_READY = True
    print("Desarrollo verificado; no se descargaron valores finales.")
else:
    print("Planificación: no se descargaron los CSV.")


## 8. Recuperar una vez y continuar la campaña

El controlador restaura los 494 resultados confirmados y verifica el plan, la autorización y el hardware antes de reclamar una nueva ejecución. `RETRY_LOST_ONCE=True` habilita exclusivamente el trabajo señalado en esa autorización, con el mismo payload y semilla. El nuevo recibo físico se conserva fuera del registro lógico original.

Si ya existe un recibo observado válido de esta recuperación, se utiliza para reconciliar el resultado sin volver a ajustar el modelo. Si hay otra interrupción sin recibo observado, se bloquea el caso para revisarlo. Después de reconciliar, la selección continúa con los candidatos pendientes y el presupuesto conservado. No se repiten los 494 trabajos anteriores y no hay ampliación automática del límite global.


In [ ]:
EXECUTION_ATTEMPTED = False
LAST_SEARCH_RETURN_CODE = None
if RUN_SEARCH:
    if not RETRY_LOST_ONCE:
        raise RuntimeError("Activa RETRY_LOST_ONCE sólo para la recuperación manual autorizada.")
    if not (SOURCE_READY and RECOVERY_SOURCE_READY and PLAN_READY and REMOTE_READY and GPU_READY and SYNTHETIC_READY and DATA_READY):
        raise RuntimeError("Completa la fuente, el plan, la persistencia, CUDA y los datos antes del lanzamiento.")
    verify_source_snapshot()
    verify_controller_snapshot()
    command = search_command(GPU_PYTHON, execute=True)
    EXECUTION_ATTEMPTED = True
    completed = subprocess.run([str(item) for item in command], cwd=RECOVERY_REPO, env=GITHUB_ENV, text=True)
    LAST_SEARCH_RETURN_CODE = completed.returncode
    print("Código de salida del ejecutor:", completed.returncode)
    print("Caché local de la campaña:", CAMPAIGN_OUTPUT)
    if completed.returncode:
        print("Revisa el estado y conserva los recibos; no borres la campaña ni repitas un intento para ocultar el fallo.")
else:
    print("Plan revisado; RUN_SEARCH=False mantiene la búsqueda sin iniciar.")


## 9. Distinguir el estado lógico del nuevo recibo físico

El estado del motor A3 cuenta los trabajos lógicos confirmados. El informe externo describe la nueva ejecución física de recuperación. Son dos evidencias diferentes y se muestran juntas, sin sustituir el resultado, el tiempo o los pasos desconocidos del intento perdido por cero.

Una nueva observación puede resolver el único trabajo pendiente, pero no convierte en conocido el intento anterior. La asignación conservadora de 8 horas corresponde a aquella sesión; el consumo observado de esta sesión se registra aparte. Si falla la confirmación de GitHub, conserva ambos estados y revisa su sincronización antes de asumir que se puede continuar.


In [ ]:
if RUN_SEARCH and EXECUTION_ATTEMPTED and CAMPAIGN_OUTPUT is not None:
    if LAST_SEARCH_RETURN_CODE != 0:
        print("El proceso no confirmó una finalización correcta. Los estados locales pueden ser anteriores o no estar confirmados en GitHub.")
    logical = CAMPAIGN_OUTPUT/"status.json"
    physical = CAMPAIGN_OUTPUT.parent/("recovery-"+EXPECTED_RECOVERY_AUTHORIZATION_SHA256[:16]+".json")
    view = {
        "version":"e2-recovery36-notebook-view-v1",
        "last_process_return_code":LAST_SEARCH_RETURN_CODE,
        "logical_campaign_status":json.loads(logical.read_text(encoding="utf-8")) if logical.is_file() else None,
        "external_physical_recovery_report":json.loads(physical.read_text(encoding="utf-8")) if physical.is_file() else None,
        "original_lost_attempt_interpretation":"Outcome, elapsed time and optimizer steps remain unknown; a zero known lower bound is not an observed zero cost.",
    }
    print(json.dumps(view, ensure_ascii=False, indent=2, allow_nan=False))
elif RUN_SEARCH:
    print("La recuperación no se lanzó. Revisa la T4, las celdas anteriores y los dos indicadores de ejecución.")
else:
    print("MODO INSPECCIÓN EN CPU: no se reclamó un intento ni se ejecutó recuperación.")
    print("RUN_SEARCH:", RUN_SEARCH, "RETRY_LOST_ONCE:", RETRY_LOST_ONCE)
